# 09_03 The bottleneck: where does one vector run out?

The encoder hands the decoder 256 numbers, whatever it read. This notebook measures what that costs: how the
lab's translator scores as sentences get longer, and what happens when you give it two sentences at once,
each of which it can translate on its own.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

In [ ]:
import collections
import json
import os
import random
import torch
import translate
from nlpcheck import ask, check_09_03, guess, reveal

torch.set_num_threads(4)
data = translate.load_split()
model = translate.lab_model()
print("the lab's translator is loaded")

## 1. Recall

**r5.** Beam search with a beam width of 1 is the same as what? (a) greedy decoding, (b) teacher forcing,
(c) no decoding at all

**r6.** Why does beam search divide each finished sentence's log-probability by a power of its length?
(a) to make it faster, (b) to use less memory, (c) because every extra word lowers the product, so without
it the shortest sentence always wins

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. Scores by sentence length

All 4,695 held-out sentences, decoded greedily in a few seconds, then scored with chrF separately for each
source length in tokens (punctuation counts, and 10 means 10 or more). Every length was trained on: the
training pairs run from two tokens to about ten. Predict: as the English gets longer, does chrF **fall**,
**rise**, or stay **flat**?

In [ ]:
guess("length_trend", None)   # "falls", "rises" or "flat" 

In [ ]:
hyps = translate.greedy(model, data["test_src"], data)
groups = collections.defaultdict(list)
for i, s in enumerate(data["test_src"]):
    groups[min(len(s), 10)].append(i)
by_length = {}
for n in sorted(groups):
    idx = groups[n]
    by_length[n] = round(translate.chrf([hyps[i] for i in idx], [data["test_refs"][i] for i in idx]), 1)
    print(f"{n:2} tokens  {len(idx):5} sentences  chrF {by_length[n]}")
trend = "falls" if by_length[9] < by_length[4] - 5 else "rises" if by_length[9] > by_length[4] + 5 else "flat"
reveal("length_trend", trend)

It falls, steadily: from about 50 at four tokens to about 35 at nine and 32 at ten or more, measured for this
lab. (The two-token row holds only four sentences and says little.) Longer sentences are also harder for a
human translator, so some fall is expected. But the shape is the one the chapter's design predicts: every
extra word has to be squeezed into the same 256 numbers, and what the decoder receives about the start of
the sentence is whatever survived nine updates of the encoder's state.

## 3. Two sentences, one vector

The sharpest test: give the translator two short sentences it handles well on its own, joined into one
input.

In [ ]:
for s in ("Tom is tired.", "Mary is hungry.", "Tom is tired. Mary is hungry."):
    print(f"{s:32} greedy: {translate.translate(model, s, data):40} beam: {translate.translate(model, s, data, k=5)}")

In the run measured for this lab, the beam gets each half right on its own ("tom está cansado ." and "mary
tiene hambre ."; greedy decoding made Mary innocent, "inocente", instead of hungry). Joined, even the beam
blends them: "tom está escribiendo un poco de hambre ." ("Tom is writing a little hunger"). Mary has gone,
and her hunger has moved to Tom. The context vector did not keep two facts about two people apart; it kept a
mixture, and the decoder wrote the mixture out.

## 4. Your turn: joined against separate, on 200 pairs

Two hundred pairs of held-out sentences of four to six tokens, drawn at random with a fixed seed. `joined`
translates each pair as one input. Complete `separate`: translate the first halves and the second halves
on their own (done for you), then join each pair of translations, `a + b`, so both are scored against the
same joined references. Predict first: how many chrF points will separate beat joined by?

In [ ]:
guess("joined_gap", None)   # a whole number of chrF points

In [ ]:
rng = random.Random(0)
pool = [i for i, s in enumerate(data["test_src"]) if 4 <= len(s) <= 6]
rng.shuffle(pool)
A, B = pool[:200], pool[200:400]
join_refs = [[data["test_refs"][a][0] + data["test_refs"][b][0]] for a, b in zip(A, B)]
joined = translate.greedy(model, [data["test_src"][a] + data["test_src"][b] for a, b in zip(A, B)], data, max_len=32)
first = translate.greedy(model, [data["test_src"][a] for a in A], data)
second = translate.greedy(model, [data["test_src"][b] for b in B], data)
separate = None    # YOUR CODE HERE: join each first translation with its second, a + b, for every pair
results = {"by_length": by_length, "joined_chrf": translate.chrf(joined, join_refs),
           "separate_chrf": translate.chrf(separate, join_refs) if separate else None}
print(results["joined_chrf"], results["separate_chrf"])
if separate:
    reveal("joined_gap", round(results["separate_chrf"] - results["joined_chrf"]))

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump(results, open("out/09_03_results.json", "w"), indent=1)
check_09_03()

About 25 points: in the run measured for this lab, 46.9 separately against 22.3 joined, from the same model on
the same sentences. Be precise about what this shows, because two things are wrong with the joined input.
It is longer than almost anything in training, and the model has never learned what to do with a second
sentence. And it has to pass through one fixed vector. This experiment cannot separate the two, and
Sutskever's team found a partial remedy for the second in reading the source backwards, so the first word
is fresh in the state when the decoder needs it. The real remedy removes the premise: let the decoder look
back at **every** encoder state, not just the last, and choose, at each word it writes, which of them to
use. That is **attention**, and it is the next lab.

## 5. Exit ticket

**x3.** Why does the joined input fail where the halves succeed? (a) the vocabulary is too small, (b) beam
search was not used, (c) both sentences must fit into the same 256 numbers as one, and no training source
was that long

In [ ]:
ask("x3", "")

Explain it back: what would the decoder need, in place of one context vector, to translate the second
sentence as well as the first?

*Your explanation:* 